# 185 — Model Compression Analysis

Analyze model compressibility: head/layer pruning candidates,
effective parameter counts, low-rank compressibility,
and overall redundancy scoring.

## Why This Matters

Model compression provides tools for systematic analysis and comparison of transformer internals. These diagnostics help identify unexpected behaviors, compare architectures, and build a comprehensive picture of how the model processes information.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.model_compression_analysis import (
    head_pruning_candidates,
    layer_pruning_candidates,
    effective_parameter_count,
    weight_low_rank_compressibility,
    redundancy_score,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = head_pruning_candidates(model, tokens, top_k=5)
print(f"Mean head norm: {result['mean_norm']:.4f}\n")
print("Top pruning candidates (lowest output norm):")
for h in result['pruning_candidates']:
    print(f"  L{h['layer']}H{h['head']}: norm={h['output_norm']:.4f}")

In [ ]:
result = layer_pruning_candidates(model, tokens)
print(f"Most skippable: Layer {result['most_skippable']}")
print(f"Least skippable: Layer {result['least_skippable']}\n")
for layer in result['per_layer']:
    bar = '#' * int(layer['relative_contribution'] * 50)
    print(f"  Layer {layer['layer']}: contrib={layer['contribution_norm']:.4f}  "
          f"relative={layer['relative_contribution']:.3f}  {bar}")

In [ ]:
result = effective_parameter_count(model, threshold=0.01)
print(f"Total params: {result['total_params']:,}")
print(f"Effective params: {result['effective_params']:,}")
print(f"Compression ratio: {result['compression_ratio']:.1%}\n")
for name, c in result['per_component'].items():
    print(f"  {name:12s}: {c['effective']:6,}/{c['total']:6,}  ({c['ratio']:.1%})")

In [ ]:
for l in range(4):
    result = weight_low_rank_compressibility(model, layer=l, target_rank=4)
    print(f"Layer {l}:")
    for name, m in result['matrices'].items():
        print(f"  {name:5s}: 90%@rank{m['rank_for_90pct']}  95%@rank{m['rank_for_95pct']}  "
              f"99%@rank{m['rank_for_99pct']}  rank4_energy={m['energy_at_target']:.3f}")
    print()

In [ ]:
result = redundancy_score(model, tokens)
print(f"Layer similarity: {result['layer_similarity']:.3f}")
print(f"Head similarity:  {result['head_similarity']:.3f}")
print(f"Redundancy score: {result['redundancy_score']:.3f}")